In [45]:
import pandas as pd
inventory = pd.DataFrame({
    "sku": ["A", "B", "C", "D", "E"],
    "stock": [12, 4, 0, 25, 7],
    "weekly_sales": [10, 8, 3, 5, 12],
    "supplier_days": [3, 5, 7, 2, 4],
})
inventory

,sku,stock,weekly_sales,supplier_days
0,A,12,10,3
1,B,4,8,5
2,C,0,3,7
3,D,25,5,2
4,E,7,12,4


# Business rules:
   * Calculate how many weeks of stock remain.
   * Reorder if remaining stock is less than supplier lead 
     time in weeks.
   * Mark zero-stock products as urgent.
   * Recommend an order quantity sufficient for four weeks of
     sales.
   * Avoid negative order quantities.​‌


In [46]:
inventory["weeks_remaining"]=(
    inventory["stock"]/inventory["weekly_sales"]
).round(2)
inventory

,sku,stock,weekly_sales,supplier_days,weeks_remaining
0,A,12,10,3,1.20
1,B,4,8,5,0.50
2,C,0,3,7,0.00
3,D,25,5,2,5.00
4,E,7,12,4,0.58


In [47]:
inventory["lead_time_weeks"]=(
    inventory["supplier_days"]/7
).round(2)
inventory

,sku,stock,weekly_sales,supplier_days,weeks_remaining,lead_time_weeks
0,A,12,10,3,1.20,0.43
1,B,4,8,5,0.50,0.71
2,C,0,3,7,0.00,1.00
3,D,25,5,2,5.00,0.29
4,E,7,12,4,0.58,0.57


In [48]:
inventory["reorder"]=inventory["lead_time_weeks"]>inventory["weeks_remaining"]
inventory

,sku,stock,weekly_sales,supplier_days,weeks_remaining,lead_time_weeks,reorder
0,A,12,10,3,1.20,0.43,False
1,B,4,8,5,0.50,0.71,True
2,C,0,3,7,0.00,1.00,True
3,D,25,5,2,5.00,0.29,False
4,E,7,12,4,0.58,0.57,False


In [49]:
inventory["urgent"]=inventory["stock"]==0
inventory["recommended_order_qty"]=(
    inventory["weekly_sales"]*4 - inventory["stock"]
).clip(lower=0)
inventory

,sku,stock,weekly_sales,supplier_days,weeks_remaining,lead_time_weeks,reorder,urgent,recommended_order_qty
0,A,12,10,3,1.20,0.43,False,False,28
1,B,4,8,5,0.50,0.71,True,False,28
2,C,0,3,7,0.00,1.00,True,True,12
3,D,25,5,2,5.00,0.29,False,False,0
4,E,7,12,4,0.58,0.57,False,False,41


In [62]:
reorder_report=inventory.loc[inventory["reorder"],["sku","recommended_order_qty","urgent"]].rename(columns={"recommended_order_qty":"quantity"})
reorder_report

,sku,quantity,urgent
1,B,28,False
2,C,12,True


In [64]:
reorder_report=reorder_report.sort_values("urgent",ascending=False)
reorder_report

,sku,quantity,urgent
2,C,12,True
1,B,28,False


### Assumptions Behind the Reorder Formula

- Future sales will occur at approximately the same rate as current `weekly_sales`.
- Demand remains stable during the planning period.
- Supplier lead time (`supplier_days`) remains consistent.
- Current stock levels are accurate and up to date.
- No additional stock is expected from other sources before the next delivery.
- Backorders, damaged inventory, and stock losses are not considered.
- The business wants to maintain approximately four weeks of inventory coverage.
- Order quantities can be adjusted freely without minimum order restrictions.
- Products can be replenished immediately once a reorder decision is made.
- Negative order quantities are not meaningful, so they are treated as zero.
- Seasonal demand fluctuations and unexpected spikes in sales are ignored.
- Each product is evaluated independently of other products.